## 1. Setup et Imports


In [ ]:
# Imports standard
import sys
import json
from pathlib import Path
from datetime import datetime
from typing import Any, Dict, List

# Imports data science
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Configuration visualisation
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('Set2')
%matplotlib inline

# Ajouter le chemin du projet
project_root = Path.cwd().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print('✓ Imports réussis')
print(f'✓ Racine du projet: {project_root}')


## 2. Configuration de l'Expérience


In [ ]:
# Chemins
RESULTS_DIR = project_root / 'experiments' / 'results' / 'rq2'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Charger un rapport RQ2 réel (généré par ConsistencyExperimentReport.save)
report_files = sorted(RESULTS_DIR.glob('rq2_report_*.json'))
if not report_files:
    raise FileNotFoundError(
        f'Aucun rapport RQ2 trouvé dans {RESULTS_DIR}. '
        'Exécute d'abord une expérience RQ2 pour générer un fichier rq2_report_*.json.'
    )

REPORT_PATH = report_files[-1]
with open(REPORT_PATH, 'r') as f:
    report: Dict[str, Any] = json.load(f)

print('📁 RQ2 report:')
print(f'  - Path: {REPORT_PATH}')
print(f"  - Experiment ID: {report.get('experiment_id')}")
print(f"  - Started: {report.get('started_at')}")
print(f"  - Completed: {report.get('completed_at')}")

cfg = report.get('config', {})
thresholds = cfg.get('thresholds', {})
print('
⚙️ Config:')
print(f"  - Models: {cfg.get('llm_models', [])}")
print(f"  - Num endpoints (config): {cfg.get('num_endpoints')}")
print(f"  - Total endpoints (report): {report.get('total_endpoints')}")
print(f"  - Min coherence score: {thresholds.get('min_coherence_score')}")


## 3. Chargement des Résultats (Réel)


In [ ]:
# Construire des DataFrames à partir du rapport JSON (réel)
endpoint_results: List[Dict[str, Any]] = report.get('endpoint_results', [])

rows: List[Dict[str, Any]] = []
inconsistency_rows: List[Dict[str, Any]] = []

for er in endpoint_results:
    counts = er.get('inconsistency_counts', {})
    flags = er.get('quality_flags', {})
    exec_meta = er.get('execution_metadata', {})

    critical = int(counts.get('critical', 0))
    major = int(counts.get('major', 0))
    minor = int(counts.get('minor', 0))
    info = int(counts.get('info', 0))
    total_inconsistencies = critical + major + minor + info

    rows.append({
        'endpoint_id': er.get('endpoint_id'),
        'endpoint_name': er.get('endpoint_name'),
        'llm_model': er.get('llm_model'),
        'coherence_score': float(er.get('coherence_score', 0.0)),
        'java_coverage_ratio': float(er.get('java_coverage_ratio', 0.0)),
        'gherkin_coverage_ratio': float(er.get('gherkin_coverage_ratio', 0.0)),
        'critical_count': critical,
        'major_count': major,
        'minor_count': minor,
        'info_count': info,
        'total_inconsistencies': total_inconsistencies,
        'missing_validations': int(counts.get('missing_validations', 0)),
        'extra_validations': int(counts.get('extra_validations', 0)),
        'incorrect_implementations': int(counts.get('incorrect_implementations', 0)),
        'passes_threshold': bool(flags.get('passes_threshold', False)),
        'has_critical_issues': bool(flags.get('has_critical_issues', False)),
        'has_major_issues': bool(flags.get('has_major_issues', False)),
        'generation_time_seconds': float(exec_meta.get('generation_time_seconds', 0.0)),
        'validation_time_seconds': float(exec_meta.get('validation_time_seconds', 0.0)),
        'error_message': exec_meta.get('error_message'),
    })

    incs_by_severity = er.get('inconsistencies', {})
    for severity, incs in incs_by_severity.items():
        for inc in incs:
            inconsistency_rows.append({
                'endpoint_id': er.get('endpoint_id'),
                'endpoint_name': er.get('endpoint_name'),
                'llm_model': er.get('llm_model'),
                'severity': severity,
                'type': inc.get('type'),
                'category': inc.get('category'),
                'field_name': inc.get('field_name'),
                'recommendation': inc.get('recommendation'),
            })

results_df = pd.DataFrame(rows)
inconsistencies_df = pd.DataFrame(inconsistency_rows)

print('📦 Données chargées:')
print(f'  - Endpoint results: {len(results_df)}')
print(f'  - Inconsistency items: {len(inconsistencies_df)}')
display(results_df.head(5))


## 4. Synthèse des Résultats (Réel)


In [ ]:
# Synthèse globale et par modèle
cfg = report.get('config', {})
thresholds = cfg.get('thresholds', {})
min_coherence = thresholds.get('min_coherence_score')

total_endpoints_reported = report.get('total_endpoints')
total_endpoints = int(total_endpoints_reported) if total_endpoints_reported is not None else int(results_df['endpoint_id'].nunique())

models = sorted([m for m in results_df['llm_model'].dropna().unique().tolist()])
print('🔍 Résumé:')
print(f'  Total endpoints (report): {total_endpoints}')
print(f'  Modèles: {models}')
print(f'  Seuil cohérence: {min_coherence}')

if results_df.empty:
    print('⚠ Aucun endpoint_result dans le rapport: analyse limitée.')
else:
    # Recalcul passes_threshold si non fourni
    if min_coherence is not None:
        results_df['passes_threshold'] = results_df['coherence_score'] >= float(min_coherence)

    by_model = (
        results_df.groupby('llm_model')
        .agg(
            endpoints=('endpoint_id', 'count'),
            coherence_mean=('coherence_score', 'mean'),
            coherence_std=('coherence_score', 'std'),
            pass_rate=('passes_threshold', 'mean'),
            critical_avg=('critical_count', 'mean'),
            major_avg=('major_count', 'mean'),
            minor_avg=('minor_count', 'mean'),
            java_cov_mean=('java_coverage_ratio', 'mean'),
            gherkin_cov_mean=('gherkin_coverage_ratio', 'mean'),
        )
        .reset_index()
        .sort_values('coherence_mean', ascending=False)
    )
    display(by_model)


## 5. Analyse des Résultats

### 5.1 Distribution des Scores de Cohérence


In [ ]:
# Distribution des scores de cohérence par modèle
if results_df.empty:
    print('⚠ Pas de données à visualiser.')
else:
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=results_df, x='llm_model', y='coherence_score')
    plt.title('RQ2: Distribution des scores de cohérence par modèle', fontsize=14, fontweight='bold')
    plt.xlabel('LLM model')
    plt.ylabel('Coherence score')
    plt.xticks(rotation=30, ha='right')

    cfg = report.get('config', {})
    thresholds = cfg.get('thresholds', {})
    min_coherence = thresholds.get('min_coherence_score')
    if min_coherence is not None:
        plt.axhline(float(min_coherence), color='red', linestyle='--', linewidth=2, label=f'Seuil {float(min_coherence):.2f}')
        plt.legend()

    out_path = RESULTS_DIR / 'rq2_coherence_distribution.png'
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'✓ Figure sauvegardée: {out_path}')


### 5.2 Distribution par Type d'Inconsistance


In [ ]:
# Analyse des inconsistances par type (réel)
if inconsistencies_df.empty:
    print('⚠ Aucune inconsistance détaillée dans le rapport.')
else:
    type_counts = (
        inconsistencies_df.groupby(['llm_model', 'type'])
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
    )
    display(type_counts.head(20))

    plt.figure(figsize=(14, 7))
    top_types = type_counts.groupby('type')['count'].sum().sort_values(ascending=False).head(10).index.tolist()
    plot_df = type_counts[type_counts['type'].isin(top_types)]
    sns.barplot(data=plot_df, x='type', y='count', hue='llm_model')
    plt.title("RQ2: Top types d'inconsistances (par modèle)", fontsize=14, fontweight='bold')
    plt.xlabel('Type')
    plt.ylabel('Nombre')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    out_path = RESULTS_DIR / 'rq2_inconsistencies_by_type.png'
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'✓ Figure sauvegardée: {out_path}')


### 5.3 Distribution par Catégorie d'Inconsistance


In [ ]:
# Analyse par catégorie (réel)
if inconsistencies_df.empty or 'category' not in inconsistencies_df.columns:
    print('⚠ Aucune catégorie disponible pour analyse.')
else:
    cat_counts = (
        inconsistencies_df.groupby(['llm_model', 'category'])
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
    )
    display(cat_counts.head(20))

    # Heatmap catégorie x modèle
    pivot = cat_counts.pivot_table(index='category', columns='llm_model', values='count', fill_value=0)
    plt.figure(figsize=(12, 8))
    sns.heatmap(pivot, annot=False, cmap='Blues')
    plt.title('RQ2: Inconsistances par catégorie et modèle', fontsize=14, fontweight='bold')
    plt.xlabel('LLM model')
    plt.ylabel('Catégorie')
    plt.tight_layout()
    out_path = RESULTS_DIR / 'rq2_inconsistencies_by_category_heatmap.png'
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'✓ Figure sauvegardée: {out_path}')


### 5.4 Relation Cohérence vs Inconsistances


In [ ]:
# Relation cohérence vs nombre d'inconsistances (réel)
if results_df.empty:
    print('⚠ Pas de données.')
else:
    plt.figure(figsize=(12, 6))
    sns.scatterplot(
        data=results_df,
        x='total_inconsistencies',
        y='coherence_score',
        hue='llm_model',
        style='llm_model',
        alpha=0.8,
    )
    plt.title("RQ2: Cohérence vs nombre total d'inconsistances", fontsize=14, fontweight='bold')
    plt.xlabel('Total inconsistances')
    plt.ylabel('Coherence score')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    out_path = RESULTS_DIR / 'rq2_coherence_vs_inconsistencies.png'
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'✓ Figure sauvegardée: {out_path}')

    corr = results_df[['total_inconsistencies', 'coherence_score']].corr().iloc[0, 1]
    print(f"
📊 Corrélation (Pearson) total_inconsistencies vs coherence_score: {corr:.4f}")


## 6. Analyse de Sévérité


In [ ]:
# Analyse par sévérité (réel)
if results_df.empty:
    print('⚠ Pas de données.')
else:
    sev_by_model = (
        results_df.groupby('llm_model')[['critical_count', 'major_count', 'minor_count', 'info_count']]
        .sum()
        .reset_index()
    )
    display(sev_by_model)

    # Stacked bar chart
    plot_df = sev_by_model.set_index('llm_model')[['critical_count', 'major_count', 'minor_count', 'info_count']]
    ax = plot_df.plot(kind='bar', stacked=True, figsize=(12, 6))
    ax.set_title("RQ2: Distribution des inconsistances par sévérité (par modèle)", fontsize=14, fontweight='bold')
    ax.set_xlabel('LLM model')
    ax.set_ylabel('Nombre total')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    out_path = RESULTS_DIR / 'rq2_severity_by_model.png'
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'✓ Figure sauvegardée: {out_path}')


## 7. Statistiques Descriptives


In [ ]:
print('📊 Statistiques descriptives (réel)
')
print('=' * 60)

if results_df.empty:
    print('⚠ Pas de données.')
else:
    # Résumé cohérence par modèle
    by_model = results_df.groupby('llm_model')['coherence_score'].describe()[['count','mean','std','min','50%','max']]
    print('
1. Cohérence par modèle:')
    display(by_model)

    corr = results_df[['total_inconsistencies', 'coherence_score']].corr().iloc[0, 1]
    print(f"
2. Corrélation Pearson (inconsistances vs cohérence): {corr:.4f}")

    cov = results_df.groupby('llm_model')[['java_coverage_ratio','gherkin_coverage_ratio']].mean()
    print('
3. Couverture moyenne (Java/Gherkin) par modèle:')
    display(cov)

print('
' + '=' * 60)


## 8. Conclusions et Recommandations


In [ ]:
print('📋 RÉSUMÉ RQ2: Validation de Cohérence Oracle ↔ Tests
')
print('=' * 60)

if results_df.empty:
    print('⚠ Rapport sans endpoint_results: rien à résumer.')
else:
    cfg = report.get('config', {})
    thresholds = cfg.get('thresholds', {})
    min_coherence = thresholds.get('min_coherence_score')
    if min_coherence is not None:
        results_df['passes_threshold'] = results_df['coherence_score'] >= float(min_coherence)

    print(f"
🎯 Modèles évalués: {sorted(results_df['llm_model'].unique().tolist())}")
    print(f"   Endpoints évalués: {len(results_df)}")
    print(f"   Seuil cohérence: {min_coherence}")

    summary = (
        results_df.groupby('llm_model')
        .agg(
            coherence_mean=('coherence_score', 'mean'),
            pass_rate=('passes_threshold', 'mean'),
            critical_total=('critical_count', 'sum'),
            major_total=('major_count', 'sum'),
            minor_total=('minor_count', 'sum'),
        )
        .reset_index()
        .sort_values('coherence_mean', ascending=False)
    )
    print('
📊 Synthèse par modèle:')
    display(summary)

    best = summary.iloc[0]
    print(f"
🏆 Meilleur modèle (cohérence moyenne): {best['llm_model']} ({best['coherence_mean']:.3f})")

print('
' + '=' * 60)


## 9. Export des Résultats


In [ ]:
# Exports (réel)
export_dir = RESULTS_DIR
export_dir.mkdir(parents=True, exist_ok=True)

results_csv = export_dir / 'rq2_endpoint_results.csv'
incs_csv = export_dir / 'rq2_inconsistencies_detailed.csv'
results_df.to_csv(results_csv, index=False)
inconsistencies_df.to_csv(incs_csv, index=False)
print(f'✓ Endpoint results exportés: {results_csv}')
print(f'✓ Inconsistances détaillées exportées: {incs_csv}')

cfg = report.get('config', {})
thresholds = cfg.get('thresholds', {})
min_coherence = thresholds.get('min_coherence_score')
if not results_df.empty and min_coherence is not None:
    results_df['passes_threshold'] = results_df['coherence_score'] >= float(min_coherence)

summary_by_model = {}
if not results_df.empty:
    for model, dfm in results_df.groupby('llm_model'):
        summary_by_model[model] = {
            'endpoints': int(len(dfm)),
            'coherence_mean': float(dfm['coherence_score'].mean()),
            'pass_rate': float(dfm['passes_threshold'].mean()) if 'passes_threshold' in dfm.columns else None,
            'critical_total': int(dfm['critical_count'].sum()),
            'major_total': int(dfm['major_count'].sum()),
            'minor_total': int(dfm['minor_count'].sum()),
            'java_coverage_mean': float(dfm['java_coverage_ratio'].mean()),
            'gherkin_coverage_mean': float(dfm['gherkin_coverage_ratio'].mean()),
        }

out_json = export_dir / 'rq2_analysis_summary.json'
payload = {
    'source_report': str(REPORT_PATH),
    'experiment_id': report.get('experiment_id'),
    'generated_at': datetime.now().isoformat(),
    'config': cfg,
    'summary_by_model': summary_by_model,
}
with open(out_json, 'w') as f:
    json.dump(payload, f, indent=2)
print(f'✓ Résumé exporté: {out_json}')

print('
✅ Analyse RQ2 (réel) terminée')
